# SNN Framework Benchmark — Colab Runner

Runs `docs/results/run_benchmark.py` (all 4 frameworks — Norse, snnTorch, SpikingJelly, Sinabs) on N-MNIST, full-scale (5 epochs, entire train/test set), one framework at a time with a cooldown gap between each so consecutive runs don't share GPU thermal/power state.

**Before you run this — read this cell.** A full run is: 4 frameworks x 5 epochs x the entire N-MNIST train set (60,000 samples) + full test set, plus 3 cooldown gaps between frameworks. Depending on the cooldown you pick, this can run for several hours. Colab's free tier disconnects idle/inactive sessions and has a total session cap (~12h) — a long `--cooldown-minutes` value increases the chance of that happening mid-run. Two things to do about it:
- Keep this browser tab open and interact with it occasionally (Colab free tier disconnects on *inactivity*, not just wall-clock time).
- If you hit disconnects, lower `COOLDOWN_MINUTES` below, or run one framework at a time across multiple sessions using `--dataset`/a single-framework filter instead of the full sweep.

Runtime -> Change runtime type -> GPU, before running anything below.

## 1. (Optional) Mount Drive, so results survive a disconnected session

In [ ]:
MOUNT_DRIVE = True  # set False to skip

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = "/content/drive/MyDrive/snn_benchmark_results"
else:
    RESULTS_ROOT = "/content/snn_benchmark_results"

import os
os.makedirs(RESULTS_ROOT, exist_ok=True)
print("Results will additionally be copied to:", RESULTS_ROOT)

## 2. Get the codebase

Clones from the repo referenced in this project's own docs (`docs/roadmap.md`). If your remote/branch differs, edit the URL/branch below before running.

In [ ]:
REPO_URL = "https://github.com/Zuzu3290/SNNs-auf-GPUs.git"
BRANCH = "main"  # change if you're running from a different branch

!git clone --branch {BRANCH} {REPO_URL} /content/SNNs-auf-GPUs
%cd /content/SNNs-auf-GPUs

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q nvidia-ml-py

import torch
print("torch:", torch.__version__, " CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — check Runtime > Change runtime type > GPU")

## 4. Run the benchmark

`--full` = 5 epochs, entire dataset (not the small diagnostic subset). TRADES is off by default (matches the plain-training comparison this run is for) — pass `--trades` here if you want adversarial training included instead.


In [ ]:
COOLDOWN_MINUTES = 20  # gap between each framework's run — see the warning in cell 1

!python docs/results/run_benchmark.py --dataset "N-MNIST" --full --cooldown-minutes {COOLDOWN_MINUTES}

## 5. Generate plots

In [ ]:
!python docs/results/make_plots.py --dataset "N-MNIST"

## 6. View the tabular results (Haseeb-style runs.csv — one row per framework)

In [ ]:
import pandas as pd

df = pd.read_csv("docs/results/data/n_mnist/runs.csv")
pd.set_option("display.max_columns", None)
df

## 7. View the plots inline

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob("docs/results/plots/n_mnist/*.png")):
    print(path)
    display(Image(filename=path))

## 8. Copy everything to Drive (survives session teardown)

In [ ]:
import shutil

if MOUNT_DRIVE:
    shutil.copytree("docs/results/data", f"{RESULTS_ROOT}/data", dirs_exist_ok=True)
    shutil.copytree("docs/results/plots", f"{RESULTS_ROOT}/plots", dirs_exist_ok=True)
    print("Copied to", RESULTS_ROOT)
else:
    print("MOUNT_DRIVE was False — results only exist in this session's local disk, "
          "download them manually before the runtime disconnects.")